In [1]:
# Code illustrating the randon walk process using the Amplitude embedding data loading strategy
# Developed by: Dr. Michael P. Haydock - IBM Fellow Emeritus, Visiting Professor at St. Olaf College
# Initial Coding: 3/4/2025

# Load the libraries
import pandas as pd  # Import pandas for handling and manipulating data
import numpy as np  # Import numpy for numerical operations
import pennylane as qml  # Import PennyLane for quantum computing functionalities

In [ ]:

# Load data
file_path = "economic_data.csv"  # File path for your dataset
data = pd.read_csv(file_path)  # Load the CSV file into a pandas DataFrame

# Drop the 'Date' column and use only numerical columns
numeric_data = data.iloc[:, 1:]  # Select all columns starting from the second column (ignores 'Date')

# Normalize the data to range [0, 1]
normalized_data = (numeric_data - numeric_data.min()) / (numeric_data.max() - numeric_data.min())
# Subtract the minimum value of each column and divide by the range (max - min) for normalization

# Validate the normalized data
if normalized_data.isnull().values.any():  # Check if there are any NaN (missing) values after normalization
    raise ValueError("Unexpected NaN values after normalization. Check numeric ranges.")  # Raise an error if found

# Quantum device setup
num_qubits = 4  # Set the number of qubits to 4 to fit small input data size (e.g., 2^4 = 16 dimensions)
dev = qml.device("default.qubit", wires=num_qubits)  # Define a quantum device using PennyLane's default qubit simulator

In [ ]:

# Quantum encoding with padding
def amplitude_encoding(data):
    """
    Encode data into quantum amplitudes with padding.
    - Pads the input data with zeros so its length matches the required size (2^num_qubits).
    - Uses AmplitudeEmbedding to map data into quantum states.
    """
    padded_data = np.pad(data, (0, 2**num_qubits - len(data)), 'constant')  # Zero-pad input to fit length 16
    qml.templates.AmplitudeEmbedding(features=padded_data, wires=range(num_qubits), normalize=True)  # Perform amplitude encoding

In [ ]:

# Quantum random walk step
def random_walk_step():
    """
    Apply a random walk using quantum gates.
    - Adds Hadamard gates to create superposition across qubits.
    - Introduces controlled rotations (CRX) for entanglement between qubits.
    """
    for wire in range(num_qubits):
        qml.Hadamard(wires=wire)  # Apply a Hadamard gate to each qubit
    for i in range(num_qubits - 1):
        qml.CRX(np.pi / 4, wires=[i, i + 1])  # Apply a controlled rotation (CRX) between consecutive qubits

In [ ]:
# Quantum node
@qml.qnode(dev)  # Define a quantum node for simulation
def random_walk(data):
    """
    Simulate a single step of the quantum random walk.
    - Encodes input data into quantum amplitudes.
    - Applies a quantum random walk step.
    - Returns the probabilities of all possible outcomes.
    """
    amplitude_encoding(data)  # Encode the input data
    random_walk_step()  # Apply the random walk step
    return qml.probs(wires=range(num_qubits))  # Return measurement probabilities of the qubits

In [ ]:
# Forecast the next 12 time periods
def forecast(data, steps=12):
    """
    Forecast the next 'steps' time periods using the quantum random walk.
    - Starts with the last row of normalized data as the initial state.
    - Iteratively performs the random walk to generate predictions.
    - Rescales predictions back to the original data range.
    """
    input_vector = data.iloc[-1].values  # Use the last row of normalized data as the starting point
    predictions = []  # Initialize an empty list to store predictions
    
    for _ in range(steps):  # Loop for the specified number of time periods
        probabilities = random_walk(input_vector)  # Perform the quantum random walk
        next_values = probabilities[:len(data.columns)]  # Use probabilities corresponding to the actual columns
        next_values = next_values * (numeric_data.max() - numeric_data.min()) + numeric_data.min()  # Rescale to original data range
        predictions.append(next_values)  # Append the predicted values to the list
        input_vector = (next_values - numeric_data.min().values) / (numeric_data.max().values - numeric_data.min().values)  
        # Normalize predictions for use as input in the next iteration

    return pd.DataFrame(predictions, columns=numeric_data.columns)  # Convert predictions into a DataFrame with column names

In [ ]:


# Generate the forecast
forecasted_data = forecast(normalized_data)  # Call the forecast function on the normalized data
forecasted_data.index = [f"Forecast {i+1}" for i in range(12)]  # Name the rows as "Forecast 1", "Forecast 2", etc.

# Display the forecasted data
print("12-Time-Period Forecast:")  # Print a header for clarity
print(forecasted_data)  # Print the forecasted data

# Generate the forecast and display the quantum circuit
forecasted_data = forecast(normalized_data)

# Visualize the quantum circuit for the first input
input_vector = normalized_data.iloc[-1].values  # Use the last row of normalized data as an example
drawer = qml.draw(random_walk)  # Create a circuit drawer
circuit_diagram = drawer(input_vector)  # Generate the circuit diagram
print(" ")
print("Quantum Circuit for the Random Walk:")
print(circuit_diagram)  # Print the circuit diagram

# Display the forecasted data
forecasted_data.index = [f"Forecast {i+1}" for i in range(12)]
print(" ")
#print("\n12-Time-Period Forecast:")
#print(forecasted_data)